# Predicting Student Test Scores 
## Score: 41.95541

In [4]:
import time
import hashlib
import numpy as np
import pandas as pd

import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

In [5]:
train = pd.read_csv('playground-series-s6e1/train.csv')

test = pd.read_csv('playground-series-s6e1/test.csv')

test_ids = test['id'].to_numpy()

y = train['exam_score'].to_numpy(dtype=float)

X = train.drop(columns=['id', 'exam_score'])
X_test = test.drop(columns=['id'])

ord_maps = {
    'sleep_quality': {'poor': 0, 'average': 1, 'good': 2},
    'facility_rating': {'low': 0, 'medium': 1, 'high': 2},
    'exam_difficulty': {'easy': 0, 'moderate': 1, 'hard': 2}
}

for col, mp in ord_maps.items():
    if col in X.columns:
        X[f'{col}_ord'] = X[col].map(mp).astype('float32')
        X_test[f'{col}_ord'] = X_test[col].map(mp).astype('float32')

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')

if 'study_hours' in X.columns and 'sleep_quality_ord' in X.columns:
    X['int_study_x_sleepq'] = X['study_hours'].astype(float) * X['sleep_quality_ord'].astype(float)
    X_test['int_study_x_sleepq'] = X_test['study_hours'].astype(float) * X_test['sleep_quality_ord'].astype(float)

if 'study_hours' in X.columns and 'class_attendance' in X.columns:
    X['int_study_x_att'] = X['study_hours'].astype(float) * X['class_attendance'].astype(float)
    X_test['int_study_x_att'] = X_test['study_hours'].astype(float) * X_test['class_attendance'].astype(float)

if 'study_hours' in X.columns and 'sleep_hours' in X.columns:
    X['int_study_x_sleep'] = X['study_hours'].astype(float) * X['sleep_hours'].astype(float)
    X_test['int_study_x_sleep'] = X_test['study_hours'].astype(float) * X_test['sleep_hours'].astype(float)

if 'sleep_hours' in X.columns:
    X['sleep_opt_dist2_8'] = (X['sleep_hours'].astype(float) - 8.0) ** 2
    X_test['sleep_opt_dist2_8'] = (X_test['sleep_hours'].astype(float) - 8.0) ** 2

for c in ['study_hours', 'sleep_hours']:
    if c in X.columns:
        X[f'log1p_{c}'] = np.log1p(X[c].astype(float))
        X_test[f'log1p_{c}'] = np.log1p(X_test[c].astype(float))

numeric_cols_to_cap = [c for c in ['study_hours', 'sleep_hours', 'class_attendance', 'age'] if c in X.columns]
for col in numeric_cols_to_cap:
    q1 = X[col].quantile(0.01)
    q99 = X[col].quantile(0.99)
    X[col] = X[col].clip(lower=q1, upper=q99)
    X_test[col] = X_test[col].clip(lower=q1, upper=q99)

bin_src_cols = [c for c in ['study_hours', 'sleep_hours', 'class_attendance'] if c in X.columns]
for c in bin_src_cols:
    _, bins = pd.qcut(X[c], q=20, duplicates='drop', retbins=True)
    bins[0] = -np.inf
    bins[-1] = np.inf
    bc = f'bin_{c}'
    X[bc] = pd.cut(X[c], bins=bins, include_lowest=True).cat.codes.astype('int16')
    X_test[bc] = pd.cut(X_test[c], bins=bins, include_lowest=True).cat.codes.astype('int16')


In [6]:
base_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'n_estimators': 8000,
    'num_leaves': 79,
    'max_depth': 10,
    'min_child_samples': 55,
    'reg_alpha': 10.0,
    'reg_lambda': 0.50,
    'min_split_gain': 1e-6,
    'subsample': 0.72,
    'subsample_freq': 3,
    'colsample_bytree': 0.65,
    'n_jobs': -1,
    'force_col_wise': True
}

seeds = [420, 666, 2025]
n_splits = 10

EARLY_STOP = 250
MAX_SECONDS = 3600

USE_CATBOOST = False

FAST_TUNE = False
FAST_TUNE_SEED = 420
FAST_TUNE_SPLITS = 5
FAST_TUNE_SMOOTHS = [5.0, 10.0, 25.0, 50.0]
FAST_TUNE_REG_ALPHA = [5.0, 10.0, 20.0]

TE_SMOOTH = 5.0
BEST_REG_ALPHA = 10.0
base_params['reg_alpha'] = BEST_REG_ALPHA

te_cols = [c for c in ['course', 'exam_difficulty', 'study_method', 'sleep_quality', 'facility_rating', 'internet_access', 'gender'] if c in X.columns]
te_cols += [c for c in X.columns if str(c).startswith('bin_')]
te_cols = list(dict.fromkeys(te_cols))

te_pairs = []
for a, b in [('course', 'exam_difficulty'), ('study_method', 'exam_difficulty'), ('course', 'study_method')]:
    if a in X.columns and b in X.columns:
        te_pairs.append((a, b))

for c in te_cols:
    vc = pd.concat([X[c], X_test[c]]).value_counts(dropna=False)
    X[f'ce_{c}'] = X[c].map(vc).astype(float).fillna(0.0)
    X_test[f'ce_{c}'] = X_test[c].map(vc).astype(float).fillna(0.0)

t0 = time.time()

alt_params = {
    **base_params,
    'num_leaves': 47,
    'max_depth': -1,
    'min_child_samples': 90,
    'reg_alpha': 20.0,
    'reg_lambda': 1.50,
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.85
}

alt_params['reg_alpha'] = BEST_REG_ALPHA * 2.0

dart_params = {
    **base_params,
    'boosting_type': 'dart',
    'n_estimators': 2500,
    'drop_rate': 0.10,
    'skip_drop': 0.50,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 30,
    'subsample': 0.80,
    'subsample_freq': 1,
    'colsample_bytree': 0.80,
    'reg_alpha': 2.0,
    'reg_lambda': 1.0
}

extra_trees_params = {
    **base_params,
    'extra_trees': True,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 30,
    'subsample': 0.90,
    'subsample_freq': 1,
    'colsample_bytree': 0.90,
    'reg_alpha': 2.0,
    'reg_lambda': 1.0
}


y_bins_cat = pd.qcut(pd.Series(y), q=20, duplicates='drop')
y_bins = y_bins_cat.cat.codes.to_numpy()
min_count = int(pd.Series(y_bins).value_counts().min())
if n_splits > min_count:
    n_splits = max(2, min_count)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

def te_fit_1_stats(X_ref, y_ref, col):
    y_s = pd.Series(y_ref, index=X_ref.index).astype(float)
    g = y_s.groupby(X_ref[col], observed=False).agg(['sum', 'count'])
    prior = float(y_s.mean())
    return g['sum'], g['count'], prior

def te_fit_2_stats(X_ref, y_ref, a, b):
    y_s = pd.Series(y_ref, index=X_ref.index).astype(float)
    key = X_ref[a].astype(str) + '|' + X_ref[b].astype(str)
    g = y_s.groupby(key).agg(['sum', 'count'])
    prior = float(y_s.mean())
    return g['sum'], g['count'], prior

def te_apply_fold_1(X_df, col, sum_s, cnt_s, prior, smooth, use_adaptive=True):
    s = X_df[col].map(sum_s)
    c = X_df[col].map(cnt_s)
    s = s.astype(float)
    c = c.astype(float)
    # Adaptive smoothing: rare categories get MORE smoothing (more conservative)
    # Common categories get LESS smoothing (more confident in their mean)
    if use_adaptive:
        smooth_adaptive = smooth * (1.0 + 3.0 / (c + 1.0))
        enc = (s + prior * smooth_adaptive) / (c + smooth_adaptive)
    else:
        enc = (s + prior * smooth) / (c + smooth)
    return enc.fillna(prior)

def te_apply_fold_2(X_df, a, b, sum_s, cnt_s, prior, smooth, use_adaptive=True):
    key = X_df[a].astype(str) + '|' + X_df[b].astype(str)
    s = key.map(sum_s)
    c = key.map(cnt_s)
    s = s.astype(float)
    c = c.astype(float)
    # Adaptive smoothing: rare pairs get MORE smoothing
    if use_adaptive:
        smooth_adaptive = smooth * (1.0 + 3.0 / (c + 1.0))
        enc = (s + prior * smooth_adaptive) / (c + smooth_adaptive)
    else:
        enc = (s + prior * smooth) / (c + smooth)
    return enc.fillna(prior)

def te_apply_loo_1(X_df, y_ref, col, sum_s, cnt_s, prior, smooth):
    y_s = pd.Series(y_ref, index=X_df.index).astype(float)
    s = X_df[col].map(sum_s).astype(float)
    c = X_df[col].map(cnt_s).astype(float)
    s2 = s - y_s
    c2 = c - 1.0
    enc = (s2 + prior * smooth) / (c2 + smooth)
    enc = enc.where(c2 > 0.0, prior)
    return enc.fillna(prior)

def te_apply_loo_2(X_df, y_ref, a, b, sum_s, cnt_s, prior, smooth):
    y_s = pd.Series(y_ref, index=X_df.index).astype(float)
    key = X_df[a].astype(str) + '|' + X_df[b].astype(str)
    s = key.map(sum_s).astype(float)
    c = key.map(cnt_s).astype(float)
    s2 = s - y_s
    c2 = c - 1.0
    enc = (s2 + prior * smooth) / (c2 + smooth)
    enc = enc.where(c2 > 0.0, prior)
    return enc.fillna(prior)

def add_te(X_tr, X_va, X_te, y_tr):
    add_tr = {}
    add_va = {}
    add_te2 = {}

    USE_ADAPTIVE_TE = True

    for c in te_cols:
        sum_s, cnt_s, prior = te_fit_1_stats(X_tr, y_tr, c)
        name = f'te_{c}'
        add_tr[name] = te_apply_fold_1(X_tr, c, sum_s, cnt_s, prior, TE_SMOOTH, USE_ADAPTIVE_TE)
        add_va[name] = te_apply_fold_1(X_va, c, sum_s, cnt_s, prior, TE_SMOOTH, USE_ADAPTIVE_TE)
        add_te2[name] = te_apply_fold_1(X_te, c, sum_s, cnt_s, prior, TE_SMOOTH, USE_ADAPTIVE_TE)

    for a, b in te_pairs:
        sum_s, cnt_s, prior = te_fit_2_stats(X_tr, y_tr, a, b)
        name = f'te_{a}__{b}'
        add_tr[name] = te_apply_fold_2(X_tr, a, b, sum_s, cnt_s, prior, TE_SMOOTH, USE_ADAPTIVE_TE)
        add_va[name] = te_apply_fold_2(X_va, a, b, sum_s, cnt_s, prior, TE_SMOOTH, USE_ADAPTIVE_TE)
        add_te2[name] = te_apply_fold_2(X_te, a, b, sum_s, cnt_s, prior, TE_SMOOTH, USE_ADAPTIVE_TE)

    X_tr2 = pd.concat([X_tr, pd.DataFrame(add_tr, index=X_tr.index)], axis=1)
    X_va2 = pd.concat([X_va, pd.DataFrame(add_va, index=X_va.index)], axis=1)
    X_te2 = pd.concat([X_te, pd.DataFrame(add_te2, index=X_te.index)], axis=1)

    return X_tr2, X_va2, X_te2

if FAST_TUNE:
    ft_bins_cat = pd.qcut(pd.Series(y), q=20, duplicates='drop')
    ft_bins = ft_bins_cat.cat.codes.to_numpy()
    ft_min_count = int(pd.Series(ft_bins).value_counts().min())
    ft_splits = FAST_TUNE_SPLITS
    if ft_splits > ft_min_count:
        ft_splits = max(2, ft_min_count)
    ft_skf = StratifiedKFold(n_splits=ft_splits, shuffle=True, random_state=123)

    best = (float('inf'), None, None)

    for sm in FAST_TUNE_SMOOTHS:
        for ra in FAST_TUNE_REG_ALPHA:
            TE_SMOOTH = float(sm)
            base_params['reg_alpha'] = float(ra)
            alt_params['reg_alpha'] = float(ra) * 2.0

            oof = np.full(len(X), np.nan, dtype=float)

            for fold, (tr_idx, va_idx) in enumerate(ft_skf.split(X, ft_bins), start=1):
                X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                y_tr, y_va = y[tr_idx], y[va_idx]

                X_tr2, X_va2, _ = add_te(X_tr, X_va, X_va, y_tr)

                p = {**base_params, 'random_state': FAST_TUNE_SEED}
                model = lgb.LGBMRegressor(**p)
                model.fit(
                    X_tr2,
                    y_tr,
                    eval_set=[(X_va2, y_va)],
                    callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)]
                )

                oof[va_idx] = model.predict(X_va2)

            filled = ~np.isnan(oof)
            rmse = float(np.sqrt(mean_squared_error(y[filled], np.clip(oof[filled], 0, 100))))

            print('FAST_TUNE', 'TE_SMOOTH', TE_SMOOTH, 'reg_alpha', base_params['reg_alpha'], 'rmse', rmse)

            if rmse < best[0]:
                best = (rmse, TE_SMOOTH, base_params['reg_alpha'])

    TE_SMOOTH = float(best[1])
    base_params['reg_alpha'] = float(best[2])
    alt_params['reg_alpha'] = float(best[2]) * 2.0
    print('FAST_TUNE BEST', 'rmse', best[0], 'TE_SMOOTH', TE_SMOOTH, 'reg_alpha', base_params['reg_alpha'])

def cv_run(params_list, seeds, label):
    sum_oof = np.zeros(len(X), dtype=float)
    cnt_oof = np.zeros(len(X), dtype=float)
    sum_test = np.zeros(len(X_test), dtype=float)
    seeds_done = 0

    for s_i, seed in enumerate(seeds, start=1):
        params_list2 = params_list if isinstance(params_list, (list, tuple)) else [params_list]

        oof = np.full(len(X), np.nan, dtype=float)
        test_pred_sum = np.zeros(len(X_test), dtype=float)
        rmse_scores = []
        folds_done = 0

        print(f'{label} SEED {seed} ({s_i}/{len(seeds)})')

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_bins), start=1):
            if (time.time() - t0) > MAX_SECONDS:
                break

            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr)

            if SELECTED_FEATURES is not None:
                common_feats = [f for f in X_tr2.columns if f in SELECTED_FEATURES]
                X_tr2 = X_tr2[common_feats]
                X_va2 = X_va2[common_feats]
                X_test2 = X_test2[common_feats]

            va_preds = []
            te_preds = []
            rmses = []

            for params in params_list2:
                p = {**params, 'random_state': seed}
                model = lgb.LGBMRegressor(**p)

                bt = str(p.get('boosting_type', 'gbdt'))
                if bt == 'dart':
                    cb = [lgb.log_evaluation(200)]
                else:
                    cb = [lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(200)]

                print('    variant', bt, 'start')

                model.fit(
                    X_tr2,
                    y_tr,
                    eval_set=[(X_va2, y_va)],
                    callbacks=cb
                )

                va_p = model.predict(X_va2)
                te_p = model.predict(X_test2)
                rmse_p = float(np.sqrt(mean_squared_error(y_va, va_p)))

                va_preds.append(va_p)
                te_preds.append(te_p)
                rmses.append(rmse_p)

            inv = 1.0 / (np.square(np.array(rmses, dtype=float)) + 1e-12)
            w = inv / inv.sum()

            va_pred = np.zeros(len(va_idx), dtype=float)
            te_pred = np.zeros(len(X_test), dtype=float)
            for wi, va_p, te_p in zip(w, va_preds, te_preds):
                va_pred += wi * va_p
                te_pred += wi * te_p

            oof[va_idx] = va_pred

            fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
            rmse_scores.append(fold_rmse)
            print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f} | w: {w.tolist()} | rmse: {rmses}')

            test_pred_sum += te_pred
            folds_done += 1

        if folds_done == 0:
            break

        test_pred = test_pred_sum / folds_done

        filled = ~np.isnan(oof)
        oof_filled = np.clip(oof[filled], 0, 100)
        sum_oof[filled] += oof_filled
        cnt_oof[filled] += 1.0

        sum_test += np.clip(test_pred, 0, 100)
        seeds_done += 1

        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], oof_filled)))
        print(f'{label} Seed {seed} OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})')

        if (time.time() - t0) > MAX_SECONDS:
            break

    denom = np.maximum(cnt_oof, 1.0)
    all_oof = np.clip(sum_oof / denom, 0, 100)

    if seeds_done > 0:
        all_test = np.clip(sum_test / seeds_done, 0, 100)
    else:
        all_test = np.zeros(len(X_test), dtype=float)

    filled_all = cnt_oof > 0
    if filled_all.any():
        final_oof_rmse = float(np.sqrt(mean_squared_error(y[filled_all], all_oof[filled_all])))
    else:
        final_oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {final_oof_rmse:.5f}')

    return all_oof, all_test, final_oof_rmse


def cv_run_cb(params, seed, label, n_splits_cb=3):
    skf_cb = StratifiedKFold(n_splits=n_splits_cb, shuffle=True, random_state=seed)

    oof = np.full(len(X), np.nan, dtype=float)
    test_pred_sum = np.zeros(len(X_test), dtype=float)
    rmse_scores = []
    folds_done = 0

    cb_cat_cols = [c for c in te_cols if c in X.columns]

    print(f'{label} SEED {seed} (1/1)')

    for fold, (tr_idx, va_idx) in enumerate(skf_cb.split(X, y_bins), start=1):
        if (time.time() - t0) > MAX_SECONDS:
            break

        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr)

        # Apply feature selection if enabled
        if SELECTED_FEATURES is not None:
            common_feats = [f for f in X_tr2.columns if f in SELECTED_FEATURES]
            X_tr2 = X_tr2[common_feats]
            X_va2 = X_va2[common_feats]
            X_test2 = X_test2[common_feats]

        cat_features = [int(X_tr2.columns.get_loc(c)) for c in cb_cat_cols if c in X_tr2.columns]

        tr_pool = Pool(X_tr2, y_tr, cat_features=cat_features)
        va_pool = Pool(X_va2, y_va, cat_features=cat_features)
        te_pool = Pool(X_test2, cat_features=cat_features)

        model = CatBoostRegressor(
            **params,
            random_seed=seed,
            allow_writing_files=False
        )

        print(f'  Fold {fold}/{n_splits_cb} start')

        model.fit(tr_pool, eval_set=va_pool, use_best_model=True, verbose=200)

        va_pred = model.predict(va_pool)
        te_pred = model.predict(te_pool)

        oof[va_idx] = va_pred
        test_pred_sum += te_pred
        folds_done += 1

        fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
        rmse_scores.append(fold_rmse)
        print(f'  Fold {fold}/{n_splits_cb} RMSE: {fold_rmse:.5f}')

    if folds_done == 0:
        all_test = np.full(len(X_test), np.nan, dtype=float)
    else:
        all_test = test_pred_sum / folds_done

    filled = ~np.isnan(oof)
    if filled.any():
        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], np.clip(oof[filled], 0, 100))))
    else:
        oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores) if rmse_scores else float("nan"):.5f} (+/- {np.std(rmse_scores) if rmse_scores else float("nan"):.5f})')

    return np.clip(oof, 0, 100), np.clip(all_test, 0, 100), oof_rmse


USE_OUTLIER_FILTER = True
OUTLIER_THRESHOLD = 3.5

if USE_OUTLIER_FILTER:
    print('OUTLIER_FILTER: fitting quick base model...')
    quick_oof = np.full(len(X), np.nan, dtype=float)
    skf_quick = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    for tr_idx, va_idx in skf_quick.split(X, y_bins):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        
        X_tr2, X_va2, _ = add_te(X_tr, X_va, X_va, y_tr)
        
        p = {**base_params, 'random_state': 42, 'n_estimators': 500}
        m = lgb.LGBMRegressor(**p)
        m.fit(X_tr2, y_tr, eval_set=[(X_va2, y_va)], 
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        quick_oof[va_idx] = m.predict(X_va2)
    
    resid = (y - quick_oof)
    resid_mean = np.nanmean(resid)
    resid_std = np.nanstd(resid)
    z_scores = np.abs((resid - resid_mean) / (resid_std + 1e-6))
    
    keep_mask = z_scores <= OUTLIER_THRESHOLD
    n_removed = int((~keep_mask).sum())
    print(f'OUTLIER_FILTER: removing {n_removed} samples ({100*n_removed/len(X):.2f}%)')
    
    X = X.loc[keep_mask].reset_index(drop=True)
    y = y[keep_mask]
    y_bins = y_bins[keep_mask]
    
    y_bins_cat = pd.qcut(pd.Series(y), q=20, duplicates='drop')
    y_bins = y_bins_cat.cat.codes.to_numpy()
    min_count = int(pd.Series(y_bins).value_counts().min())
    if n_splits > min_count:
        n_splits = max(2, min_count)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

USE_FEATURE_SELECT = True
FEATURE_KEEP_PCT = 0.90

if USE_FEATURE_SELECT:
    print('FEATURE_SELECT: computing OOF importance...')
    imp_oof = np.full(len(X), np.nan, dtype=float)
    imp_scores = {}
    skf_imp = StratifiedKFold(n_splits=3, shuffle=True, random_state=123)
    
    for tr_idx, va_idx in skf_imp.split(X, y_bins):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        
        X_tr2, X_va2, _ = add_te(X_tr, X_va, X_va, y_tr)
        
        p = {**base_params, 'random_state': 123, 'n_estimators': 500}
        m = lgb.LGBMRegressor(**p)
        m.fit(X_tr2, y_tr, eval_set=[(X_va2, y_va)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        
        for feat, imp in zip(X_tr2.columns, m.feature_importances_):
            imp_scores[feat] = imp_scores.get(feat, 0.0) + imp / 3.0
    
    imp_df = pd.Series(imp_scores).sort_values(ascending=False)
    n_keep = max(10, int(len(imp_df) * FEATURE_KEEP_PCT))
    keep_feats = imp_df.head(n_keep).index.tolist()
    
    print(f'FEATURE_SELECT: keeping {len(keep_feats)}/{len(imp_df)} features')
    
    SELECTED_FEATURES = set(keep_feats)
else:
    SELECTED_FEATURES = None

USE_RANKING_BLEND = True

if USE_RANKING_BLEND:
    print('RANKING_BLEND: training multiple variants separately...')
    
    variant_oofs = []
    variant_tests = []
    variant_names = []
    
    oof1, test1, _ = cv_run([base_params], seeds, 'LGB_BASE')
    variant_oofs.append(oof1)
    variant_tests.append(test1)
    variant_names.append('base')
    
    oof2, test2, _ = cv_run([alt_params], seeds, 'LGB_ALT')
    variant_oofs.append(oof2)
    variant_tests.append(test2)
    variant_names.append('alt')
    
    variant3_params = {
        **base_params,
        'num_leaves': 63,
        'min_child_samples': 100,
        'reg_alpha': 15.0,
        'reg_lambda': 1.0,
        'subsample': 0.80,
        'colsample_bytree': 0.70
    }
    oof3, test3, _ = cv_run([variant3_params], seeds, 'LGB_REG')
    variant_oofs.append(oof3)
    variant_tests.append(test3)
    variant_names.append('reg')
    
    print('RANKING_BLEND: training XGBoost variant...')
    
    def cv_run_xgb(params, seeds, label):
        sum_oof = np.zeros(len(X), dtype=float)
        cnt_oof = np.zeros(len(X), dtype=float)
        sum_test = np.zeros(len(X_test), dtype=float)
        seeds_done = 0
        
        for s_i, seed in enumerate(seeds, start=1):
            oof = np.full(len(X), np.nan, dtype=float)
            test_pred_sum = np.zeros(len(X_test), dtype=float)
            rmse_scores = []
            folds_done = 0
            
            print(f'{label} SEED {seed} ({s_i}/{len(seeds)})')
            
            for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_bins), start=1):
                if (time.time() - t0) > MAX_SECONDS:
                    break
                
                X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                y_tr, y_va = y[tr_idx], y[va_idx]
                
                X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr)
                
                if SELECTED_FEATURES is not None:
                    common_feats = [f for f in X_tr2.columns if f in SELECTED_FEATURES]
                    X_tr2 = X_tr2[common_feats]
                    X_va2 = X_va2[common_feats]
                    X_test2 = X_test2[common_feats]
                
                model = xgb.XGBRegressor(
                    **params,
                    random_state=seed,
                    tree_method='hist',
                    n_jobs=-1,
                    enable_categorical=True
                )
                
                model.fit(
                    X_tr2, y_tr,
                    eval_set=[(X_va2, y_va)],
                    verbose=False
                )
                
                va_p = model.predict(X_va2)
                te_p = model.predict(X_test2)
                
                oof[va_idx] = va_p
                test_pred_sum += te_p
                folds_done += 1
                
                fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_p)))
                rmse_scores.append(fold_rmse)
                print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f}')
            
            if folds_done == 0:
                break
            
            test_pred = test_pred_sum / folds_done
            
            filled = ~np.isnan(oof)
            oof_filled = np.clip(oof[filled], 0, 100)
            sum_oof[filled] += oof_filled
            cnt_oof[filled] += 1.0
            
            sum_test += np.clip(test_pred, 0, 100)
            seeds_done += 1
            
            oof_rmse = float(np.sqrt(mean_squared_error(y[filled], oof_filled)))
            print(f'{label} Seed {seed} OOF RMSE: {oof_rmse:.5f}')
            
            if (time.time() - t0) > MAX_SECONDS:
                break
        
        denom = np.maximum(cnt_oof, 1.0)
        all_oof = np.clip(sum_oof / denom, 0, 100)
        all_test = np.clip(sum_test / seeds_done, 0, 100) if seeds_done > 0 else np.zeros(len(X_test), dtype=float)
        
        filled_all = cnt_oof > 0
        final_oof_rmse = float(np.sqrt(mean_squared_error(y[filled_all], all_oof[filled_all]))) if filled_all.any() else float('nan')
        
        print(f'{label} FINAL OOF RMSE: {final_oof_rmse:.5f}')
        return all_oof, all_test, final_oof_rmse
    
    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': 0.03,
        'n_estimators': 8000,
        'max_depth': 8,
        'min_child_weight': 55,
        'reg_alpha': 10.0,
        'reg_lambda': 0.50,
        'subsample': 0.72,
        'colsample_bytree': 0.65,
        'gamma': 1e-6
    }
    
    oof4, test4, _ = cv_run_xgb(xgb_params, seeds, 'XGB')
    variant_oofs.append(oof4)
    variant_tests.append(test4)
    variant_names.append('xgb')
    
    variant_oofs = np.array(variant_oofs)
    variant_tests = np.array(variant_tests)
    
    print(f'RANKING_BLEND: collected {len(variant_names)} variants: {variant_names}')
    
    # --- Ranking-based blending function ---
    def ranking_blend(predictions, main_weights, position_weights, blend_asc_weight=0.70):
        """
        Blend predictions using ranking-based weighting.
        
        Args:
            predictions: array of shape (n_variants, n_samples) - predictions from each variant
            main_weights: list of base weights for each variant (e.g., [0.5, 0.3, 0.2])
            position_weights: list of position bonuses (e.g., [0.05, -0.01, -0.04])
            blend_asc_weight: weight for ascending blend (descending gets 1 - this)
        
        Returns:
            blended predictions of shape (n_samples,)
        """
        n_variants, n_samples = predictions.shape
        blended = np.zeros(n_samples, dtype=float)
        
        main_weights = np.array(main_weights, dtype=float)
        main_weights = main_weights / main_weights.sum()
        
        # For each sample, sort predictions and apply position-dependent weights
        for i in range(n_samples):
            preds = predictions[:, i]
            
            # Ascending blend: sort low to high
            asc_order = np.argsort(preds)
            asc_weights = main_weights[asc_order] + np.array(position_weights)
            asc_weights = np.maximum(asc_weights, 0.0)  # ensure non-negative
            asc_weights = asc_weights / asc_weights.sum()
            asc_blend = np.sum(preds[asc_order] * asc_weights)
            
            # Descending blend: sort high to low
            desc_order = np.argsort(preds)[::-1]
            desc_weights = main_weights[desc_order] + np.array(position_weights)
            desc_weights = np.maximum(desc_weights, 0.0)
            desc_weights = desc_weights / desc_weights.sum()
            desc_blend = np.sum(preds[desc_order] * desc_weights)
            
            # Combine ascending and descending blends
            blended[i] = blend_asc_weight * asc_blend + (1.0 - blend_asc_weight) * desc_blend
        
        return blended
    
    main_wts = [0.40, 0.25, 0.20, 0.15]
    pos_wts = [0.06, 0.00, -0.02, -0.04]
    
    asc_wt = 0.70
    
    blend_oof = ranking_blend(variant_oofs, main_wts, pos_wts, asc_wt)
    blend_oof = np.clip(blend_oof, 0, 100)
    
    blend_test = ranking_blend(variant_tests, main_wts, pos_wts, asc_wt)
    blend_test = np.clip(blend_test, 0, 100)
    
    blend_rmse = float(np.sqrt(mean_squared_error(y, blend_oof)))
    print(f'RANKING_BLEND OOF RMSE: {blend_rmse:.5f}')
    print(f'  Individual OOF RMSEs:')
    for name, oof_pred in zip(variant_names, variant_oofs):
        ind_rmse = float(np.sqrt(mean_squared_error(y, np.clip(oof_pred, 0, 100))))
        print(f'    {name}: {ind_rmse:.5f}')
    
    lgb_oof = blend_oof
    lgb_test = blend_test
else:
    # Fallback to original blending
    lgb_oof, lgb_test, _ = cv_run([base_params, alt_params], seeds, 'LGB2')

if USE_CATBOOST:
    cb_params = {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'iterations': 2500,
        'learning_rate': 0.03,
        'depth': 8,
        'l2_leaf_reg': 6.0,
        'random_strength': 1.0,
        'bagging_temperature': 0.5,
        'subsample': 0.80,
        'rsm': 0.85,
        'min_data_in_leaf': 25,
        'od_type': 'Iter',
        'od_wait': 100,
        'thread_count': -1
    }

    cb_oof, cb_test, _ = cv_run_cb(cb_params, seed=2025, label='CAT', n_splits_cb=3)

    cb_oof2 = np.array(cb_oof, dtype=float, copy=True)
    cb_test2 = np.array(cb_test, dtype=float, copy=True)

    m_oof = np.isnan(cb_oof2)
    if m_oof.any():
        cb_oof2[m_oof] = lgb_oof[m_oof]

    m_test = np.isnan(cb_test2)
    if m_test.any():
        cb_test2[m_test] = lgb_test[m_test]

    blend_X = np.vstack([lgb_oof, cb_oof2]).T
    blend_lr = LinearRegression(positive=True)
    blend_lr.fit(blend_X, y)

    blend_oof = blend_lr.predict(blend_X)
    blend_rmse = float(np.sqrt(mean_squared_error(y, blend_oof)))
    print('BLEND coef', blend_lr.coef_.tolist(), 'intercept', float(blend_lr.intercept_), 'rmse', blend_rmse)

    pred = blend_lr.predict(np.vstack([lgb_test, cb_test2]).T)
    pred = np.clip(pred, 0, 100)
else:
    pred = np.clip(lgb_test, 0, 100)

# --- OOF residual group-bias correction (cheap “data edge”) ---
# This tries to remove systematic per-group under/over prediction using ONLY OOF residuals.
USE_RESIDUAL_CORR = True
RESID_SMOOTH = 200.0

if USE_RESIDUAL_CORR:
    base_oof = np.clip(lgb_oof, 0, 100)
    base_test = np.clip(pred, 0, 100)
    resid = (y - base_oof).astype(float)

    def apply_resid_corr(key_tr, key_te, resid_tr, smooth):
        s = pd.Series(resid_tr).groupby(key_tr, observed=False).sum()
        c = pd.Series(resid_tr).groupby(key_tr, observed=False).count()
        corr_tr = key_tr.map(s) / (key_tr.map(c) + smooth)
        corr_te = key_te.map(s) / (key_te.map(c) + smooth)
        return corr_tr.fillna(0.0).to_numpy(dtype=float), corr_te.fillna(0.0).to_numpy(dtype=float)

    corr_oof = np.zeros(len(X), dtype=float)
    corr_test = np.zeros(len(X_test), dtype=float)

    # single-group corrections
    for g in [c for c in ['course', 'study_method', 'exam_difficulty', 'sleep_quality', 'facility_rating', 'internet_access', 'gender'] if c in X.columns]:
        tr_key = X[g].astype(str)
        te_key = X_test[g].astype(str)
        co, ct = apply_resid_corr(tr_key, te_key, resid, RESID_SMOOTH)
        corr_oof += co
        corr_test += ct

    # high-signal pair corrections
    for a, b in [("course", "exam_difficulty"), ("course", "study_method"), ("study_method", "exam_difficulty")]:
        if a in X.columns and b in X.columns:
            tr_key = (X[a].astype(str) + '|' + X[b].astype(str))
            te_key = (X_test[a].astype(str) + '|' + X_test[b].astype(str))
            co, ct = apply_resid_corr(tr_key, te_key, resid, RESID_SMOOTH)
            corr_oof += co
            corr_test += ct

    # apply correction
    adj_oof = np.clip(base_oof + corr_oof, 0, 100)
    adj_rmse = float(np.sqrt(mean_squared_error(y, adj_oof)))
    print('RESIDUAL_CORR rmse', adj_rmse, 'corr_oof_std', float(np.std(corr_oof)), 'corr_test_std', float(np.std(corr_test)))

    pred = np.clip(base_test + corr_test, 0, 100)

submission = pd.DataFrame({'id': test_ids, 'exam_score': pred})

out_path = 'submission.csv'
submission.to_csv(out_path, index=False)
with open(out_path, 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print(out_path)
print('md5', md5)
print('pred_mean', float(np.mean(pred)), 'pred_std', float(np.std(pred)), 'pred_min', float(np.min(pred)), 'pred_max', float(np.max(pred)))
print('pred_ge_99_5', int(np.sum(pred >= 99.5)), 'pred_ge_97', int(np.sum(pred >= 97.0)))


OUTLIER_FILTER: fitting quick base model...
Training until validation scores don't improve for 50 rounds


Did not meet early stopping. Best iteration is:
[500]	valid_0's rmse: 8.76993
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's rmse: 8.77695
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's rmse: 8.78132
OUTLIER_FILTER: removing 398 samples (0.06%)
FEATURE_SELECT: computing OOF importance...
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's rmse: 8.75099
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[498]	valid_0's rmse: 8.73
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's rmse: 8.73515
FEATURE_SELECT: keeping 41/46 features
RANKING_BLEND: training multiple variants separately...
LGB_BASE SEED 420 (1/3)
    variant gbdt start